In [1]:
import requests
import re

In [2]:
BASE_URL = "https://rest.uniprot.org"


def parse_fasta_sequence(fasta_text: str) -> str:

    lines = fasta_text.strip().splitlines()
    seq_lines = [line.strip() for line in lines if not line.startswith(">")]
    return "".join(seq_lines)


def apply_mutations(sequence: str, region_start: int, mutations: list[str]) -> str:
    """
    Apply mutations like ['Q491K', 'I497V'] to a sequence fragment.

    sequence: the fragment sequence
    region_start: biological start position of the fragment (1-based, inclusive)
    mutations: list of strings like 'Q491K'
    """
    seq = list(sequence)

    for mut in mutations:
        mut = mut.strip()
        m = re.fullmatch(r"([A-Z])(\d+)([A-Z])", mut)
        if not m:
            raise ValueError(f"Bad mutation format: {mut}")

        original_aa, pos_str, new_aa = m.groups()
        protein_pos = int(pos_str)

        # Convert protein coordinate to fragment index
        idx = protein_pos - region_start

        if idx < 0 or idx >= len(seq):
            raise ValueError(
                f"Mutation {mut} is outside fragment range starting at {region_start}"
            )

        # Optional safety check
        if seq[idx] != original_aa:
            raise ValueError(
                f"Mutation {mut} does not match fragment: expected {original_aa}, found {seq[idx]}"
            )

        seq[idx] = new_aa

    return "".join(seq)


def search_uniprot_accession(
    protein_name: str,
    organism: str | None = None,
    reviewed: bool = True,
    size: int = 5
) -> dict:
    """
    Search UniProt by protein name (+ optional organism) and return the top hit.
    """
    query_parts = [f'"{protein_name}"']
    if organism:
        query_parts.append(f'organism_name:"{organism}"')
    if reviewed:
        query_parts.append("reviewed:true")

    query = " AND ".join(query_parts)

    url = f"{BASE_URL}/uniprotkb/search"
    params = {
        "query": query,
        "format": "json",
        "size": size,
    }

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    results = data.get("results", [])
    if not results:
        raise LookupError(f"No UniProt hit found for query: {query}")

    # Very simple strategy: use the top hit.
    # For production, you would score multiple candidates.
    top = results[0]

    accession = top["primaryAccession"]
    return {
        "accession": accession,
        "query_used": query,
        "raw_result": top,
    }


def fetch_fasta_by_accession(accession: str) -> str:
    """
    Fetch the FASTA for a UniProt accession and return the full amino acid sequence.
    """
    url = f"{BASE_URL}/uniprotkb/{accession}"
    params = {"format": "fasta"}

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()

    return parse_fasta_sequence(r.text)


def get_protein_sequence(
    protein_name: str,
    organism: str | None = None,
    region: tuple[int, int] | None = None,
    mutations: list[str] | None = None,
    reviewed: bool = True
) -> dict:
    """
    Full pipeline:
    1. Search UniProt by protein name (+ organism)
    2. Fetch canonical FASTA sequence by accession
    3. Slice a region if requested
    4. Apply mutations if requested

    region = (start, end) uses biological numbering, inclusive
    mutations = ['Q491K', 'I497V', ...]
    """
    hit = search_uniprot_accession(
        protein_name=protein_name,
        organism=organism,
        reviewed=reviewed
    )

    accession = hit["accession"]
    full_sequence = fetch_fasta_by_accession(accession)

    # Default: whole sequence
    selected_sequence = full_sequence
    region_start = 1
    region_end = len(full_sequence)

    if region is not None:
        start, end = region
        if start < 1 or end < start:
            raise ValueError(f"Invalid region: {region}")

        # Python slice: end is exclusive, biology ranges are inclusive
        selected_sequence = full_sequence[start - 1:end]
        region_start = start
        region_end = end

    mutated_sequence = selected_sequence
    if mutations:
        mutated_sequence = apply_mutations(
            sequence=selected_sequence,
            region_start=region_start,
            mutations=mutations
        )

    return {
        "protein_name_input": protein_name,
        "organism_input": organism,
        "accession": accession,
        "query_used": hit["query_used"],
        "full_sequence": full_sequence,
        "region": (region_start, region_end),
        "selected_sequence": selected_sequence,
        "mutated_sequence": mutated_sequence,
        "mutations_applied": mutations or [],
    }

In [5]:
result = get_protein_sequence(
    protein_name="Gag-Pol polyprotein",
    organism="Human immunodeficiency virus 1",
    region=(484, 582),
    mutations=[
        "Q491K", "I497V", "V499I", "E519D",
        "G532V", "I538V", "R541K", "D544E",
        "L547P", "Q553K", "V566A", "L573M"
    ]
)

print("UniProt accession:", result["accession"])
print("Region:", result["region"])
print("Selected fragment:", result["selected_sequence"])
print("Mutated fragment:", result["mutated_sequence"])

LookupError: No UniProt hit found for query: "Gag-Pol polyprotein" AND organism_name:"Human immunodeficiency virus 1" AND reviewed:true